<a href="https://colab.research.google.com/github/surajgade17/solubility-prediction-delaney/blob/main/solubility_prediction_delaney.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# Load Delaney solubility dataset
!pip install rdkit -q
import pandas as pd
url = "https://raw.githubusercontent.com/deepchem/deepchem/master/datasets/delaney-processed.csv"
df = pd.read_csv(url)
df = df[["smiles", "measured log solubility in mols per litre"]]
df.columns = ["smiles", "logS"]
print(df.shape)
df.head()

(1128, 2)


,smiles,logS
0,OCC3OC(OCC2OC(OC(C#N)c1ccccc1)C(O)C(O)C2O)C(O)...,-0.77
1,Cc1occc1C(=O)Nc2ccccc2,-3.30
2,CC(C)=CCCC(C)=CC(=O),-2.06
3,c1ccc2c(c1)ccc3c2ccc4c5ccccc5ccc43,-7.87
4,c1ccsc1,-1.33


In [5]:
# Validate SMILES and calculate descriptors with RDKit
from rdkit import Chem
from rdkit.Chem import Descriptors

def feats(smi):
    m = Chem.MolFromSmiles(smi)
    if m is None:
        return None
    return [Descriptors.MolWt(m), Descriptors.MolLogP(m),
            Descriptors.NumHDonors(m), Descriptors.NumHAcceptors(m),
            Descriptors.TPSA(m)]

df["f"] = df["smiles"].apply(feats)
print("Invalid SMILES:", df["f"].isna().sum())
df = df.dropna()
X = pd.DataFrame(df["f"].tolist(),
    columns=["MolWt","LogP","HDonors","HAcceptors","TPSA"])
y = df["logS"].values

Invalid SMILES: 0


In [6]:
# Train Random Forest and evaluate model
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestRegressor(random_state=42).fit(Xtr, ytr)
pred = model.predict(Xte)
print("R2:", round(r2_score(yte, pred), 3))
print("RMSE:", round(mean_squared_error(yte, pred) ** 0.5, 3))

R2: 0.851
RMSE: 0.838
